In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Notebook 1: World State Simulation
# MAGIC Synthetic data representing IoT sensor and weather API reads for a Downtown Seattle store
# MAGIC over a 90-minute window. Each write is tagged with `userMetadata` so Notebook 2 can
# MAGIC resolve Delta versions dynamically without hardcoding.

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, TimestampType
)
from datetime import datetime

# COMMAND ----------

# Schema
# event_timestamp : when the reading occurred in the real world
# source          : originating system (iot_sensor / weather_api)

world_state_schema = StructType([
    StructField("store",               StringType()),
    StructField("inventory_gallons",   IntegerType()),
    StructField("weather_temp_f",      IntegerType()),
    StructField("weather_condition",   StringType()),
    StructField("event_timestamp",     TimestampType()),
    StructField("source",              StringType())
])

spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.supply_agent_demo")
spark.sql("DROP TABLE IF EXISTS workspace.supply_agent_demo.world_state")

# COMMAND ----------

# Version 0: Baseline readings — normal conditions
# 07:00 to 08:00, inventory healthy, weather rainy and cool
# userMetadata tag: "baseline_normal"

baseline_readings = [
    ("Downtown", 120, 45, "Rainy",  datetime(2026, 5, 19, 7,  0), "iot_sensor"),
    ("Downtown", 115, 45, "Rainy",  datetime(2026, 5, 19, 7, 30), "iot_sensor"),
    ("Downtown", 110, 46, "Cloudy", datetime(2026, 5, 19, 8,  0), "iot_sensor"),
]

df_baseline = spark.createDataFrame(baseline_readings, world_state_schema)

df_baseline.write \
    .format("delta") \
    .option("userMetadata", "baseline_normal") \
    .mode("overwrite") \
    .saveAsTable("workspace.supply_agent_demo.world_state")

print("Baseline written. Delta version: 0")

# COMMAND ----------

# Version 1: Corrupted readings at 08:12
# Faulty IoT sensor reports inventory as -999
# Weather API glitch pushes 102F heatwave warning
# Both self-correct by 08:16 — live table will show no trace
# userMetadata tag: "corrupted_sensor"

corrupted_readings = [
    ("Downtown", -999, 102, "Extreme Heatwave",
     datetime(2026, 5, 19, 8, 12), "iot_sensor_faulty"),
]

df_corrupted = spark.createDataFrame(corrupted_readings, world_state_schema)

df_corrupted.write \
    .format("delta") \
    .option("userMetadata", "corrupted_sensor") \
    .mode("append") \
    .saveAsTable("workspace.supply_agent_demo.world_state")

print("Corrupted readings injected. Delta version: 1")

# COMMAND ----------

# Version 2: Self-corrected readings at 08:16
# APIs resolve the glitch and push accurate values
# This is what the live table shows — no trace of corruption
# userMetadata tag: "self_corrected"

corrected_readings = [
    ("Downtown", 118, 45, "Rainy",
     datetime(2026, 5, 19, 8, 16), "iot_sensor"),
]

df_corrected = spark.createDataFrame(corrected_readings, world_state_schema)

df_corrected.write \
    .format("delta") \
    .option("userMetadata", "self_corrected") \
    .mode("append") \
    .saveAsTable("workspace.supply_agent_demo.world_state")

print("Corrected readings written. Delta version: 2")

# COMMAND ----------

# Verification
# Live table should show 5 rows including the -999 anomaly
# Delta history should show 3 versions with metadata tags

print("Live table (what anyone looking right now would see):")
spark.sql("""
    SELECT store, inventory_gallons, weather_temp_f,
           weather_condition, event_timestamp, source
    FROM workspace.supply_agent_demo.world_state
    WHERE store = 'Downtown'
    ORDER BY event_timestamp DESC
""").show()

print("Delta history with userMetadata tags:")
spark.sql("""
    DESCRIBE HISTORY workspace.supply_agent_demo.world_state
""").select("version", "timestamp", "operation", "userMetadata").show(truncate=False)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Sensor Timeline Visualization
# MAGIC The chart below shows inventory levels and temperature over the 90-minute window.
# MAGIC The corrupted reading at 08:12 is immediately visible as a sharp anomaly.
# MAGIC This is what was written to the Delta table — and what the agent would read.

# COMMAND ----------

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

df_timeline = spark.sql("""
    SELECT event_timestamp, inventory_gallons, weather_temp_f,
           weather_condition, source
    FROM workspace.supply_agent_demo.world_state
    WHERE store = 'Downtown'
    ORDER BY event_timestamp ASC
""").toPandas()

df_timeline["event_timestamp"] = pd.to_datetime(df_timeline["event_timestamp"])
df_timeline["time_label"] = df_timeline["event_timestamp"].dt.strftime("%H:%M")

is_faulty = df_timeline["source"] == "iot_sensor_faulty"

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
fig.suptitle(
    "Downtown Seattle Store — Sensor Readings: 07:00 to 08:20\n"
    "Corrupted window at 08:12 is invisible in the live table without Time Travel",
    fontsize=13, fontweight="bold", y=0.98
)

# --- Inventory chart ---
bar_colors = ["#d62728" if faulty else "#1f77b4" for faulty in is_faulty]
ax1.bar(
    df_timeline["time_label"],
    df_timeline["inventory_gallons"],
    color=bar_colors,
    width=0.5,
    edgecolor="white"
)
ax1.axhline(y=0, color="#d62728", linestyle="--", linewidth=1, alpha=0.6,
            label="Zero inventory threshold")
ax1.axhline(y=50, color="#ff7f0e", linestyle="--", linewidth=1, alpha=0.6,
            label="Reorder threshold (50 gal)")
ax1.set_ylabel("Inventory (gallons)", fontsize=11)
ax1.set_title("Oat Milk Inventory", fontsize=11)
ax1.set_ylim(-1100, 200)
ax1.legend(fontsize=9, loc="upper right")

normal_patch = mpatches.Patch(color="#1f77b4", label="Normal reading")
faulty_patch = mpatches.Patch(color="#d62728", label="Corrupted reading (iot_sensor_faulty)")
ax1.legend(handles=[normal_patch, faulty_patch], fontsize=9, loc="lower right")

for i, (val, faulty) in enumerate(zip(df_timeline["inventory_gallons"], is_faulty)):
    ax1.text(i, val + 5 if val >= 0 else val - 60,
             str(val), ha="center", va="bottom", fontsize=9,
             color="#d62728" if faulty else "#1f77b4", fontweight="bold")

# --- Temperature chart ---
temp_colors = ["#d62728" if faulty else "#2ca02c" for faulty in is_faulty]
ax2.plot(df_timeline["time_label"], df_timeline["weather_temp_f"],
         color="#2ca02c", linewidth=2, marker="o", markersize=6, label="Temperature (F)")
ax2.scatter(
    df_timeline.loc[is_faulty, "time_label"],
    df_timeline.loc[is_faulty, "weather_temp_f"],
    color="#d62728", zorder=5, s=100, label="Corrupted reading"
)
ax2.axhline(y=90, color="#d62728", linestyle="--", linewidth=1, alpha=0.6,
            label="Emergency threshold (90F)")
ax2.set_ylabel("Temperature (F)", fontsize=11)
ax2.set_xlabel("Time", fontsize=11)
ax2.set_title("Weather Temperature", fontsize=11)
ax2.legend(fontsize=9, loc="upper right")

for i, (val, faulty) in enumerate(zip(df_timeline["weather_temp_f"], is_faulty)):
    ax2.text(i, val + 1.5, f"{val}F", ha="center", va="bottom", fontsize=9,
             color="#d62728" if faulty else "#2ca02c", fontweight="bold")

plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("/tmp/sensor_timeline.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved to /tmp/sensor_timeline.png")